In [ ]:
import sys
from pathlib import Path
import torch

# Resolve project root directory from notebook location
notebook_path = Path.cwd()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.evaluation.language_eval import LanguageAphasiaEvaluator
from src.models.loader import ModelEnvironmentLoader
from utils.memory import flush_memory, print_vram_usage

print("Environment setup completed. Ready to load components.")

In [ ]:
# Configuration block targeting minimal VRAM footprint
language_config = {
    "model": {"model_name": "gpt2", "torch_dtype": "bfloat16", "device": "cuda"}
}

# Load inside the isolated environment
loader = ModelEnvironmentLoader(language_config)
model = loader.load_language_model()
evaluator = LanguageAphasiaEvaluator(model)

print_vram_usage()

In [ ]:
data_json_path = project_root / "data" / "text_samples.json"

print("--- Running Baseline Evaluation (Healthy State) ---")
baseline_metrics = evaluator.run_batch_evaluation(data_json_path)

In [ ]:
# Select target circuit parameters to damage
target_layer = 5
target_head = 8

print(
    f"Simulating localized stroke at Layer {target_layer}, Head {target_head}"
)


def head_ablation_hook(value: torch.Tensor, hook) -> torch.Tensor:
    """Ablate specific head dimension during forward pass."""
    value[:, :, target_head, :] = 0.0
    return value


# Inject hook into attention structure
model.add_hook(f"blocks.{target_layer}.attn.hook_z", head_ablation_hook)
print("Stroke simulation hook is now active inside the model neural net.")

In [ ]:
print("--- Running Post-Stroke Evaluation (Lesioned State) ---")
post_stroke_metrics = evaluator.run_batch_evaluation(data_json_path)

# Print direct comparison for the first sample
if baseline_metrics and post_stroke_metrics:
    base_diff = baseline_metrics[0]["logit_difference"]
    stroke_diff = post_stroke_metrics[0]["logit_difference"]
    print(f"\nSample 1 Circuit Degradation: {base_diff:.4f} -> {stroke_diff:.4f}")

In [ ]:
print("Resetting all neural network hooks and freeing device allocation...")
model.reset_hooks()

# Explicitly delete objects and empty cache
del model
del evaluator
flush_memory()
print_vram_usage()